# 28 · Set 10: a self-learnt rulebook for model switching (A100 for the planner)

Needs notebook 27 to be finished: this notebook uses only outputs it saved (every retriever's shortlist, Llama 8B and Qwen 32B
answers, Qwen samples, on 1,000 dev questions and the 400 test questions). No training. The only GPU work is the planner.

| Part | What it is |
|---|---|
| Soft rules (start) | R1: 0.5B unsure (lowest 20%) → escalate retrieval to the 0.8B, skipping the 0.6B, and ensemble. R2: still split and unsure → escalate again. A1: answer not yes/no and not fully found in the passages → Qwen 32B answers; agree → accept, else Qwen samples vote. A2–A4: keep. |
| Hard rules (locked) | H1: 0.5B in its lowest 10% → always escalate. H2: empty / "cannot be determined" answer → always escalate. H3: at most 60% reach Qwen 32B. H4: dev accuracy ≥ best fixed pipeline − 1 point (below it, the loop is in repair mode). H5: "Sure" needs a score ≥ 0.85. H6: ≤ 8 soft rules, each firing on ≥ 20 dev questions. H7: a threshold moves ≤ 0.05 per round. H8: an accuracy gain may cost ≤ 10% compute per point. |
| Micro-experiments (dev) | 4 rounds. Code tries fine threshold steps; the planner (Qwen 32B) reads evidence tables from dev half A and proposes up to 3 changes. A change is kept only if it passes on both halves, beats random escalation of the same size, and breaks no hard rule. |
| Test | The final rulebook, frozen, applied by code once to the 400 test questions; which rule decided is logged per question, with a Sure / Unsure flag |

**My estimate:** copying Qwen 32B and installing vLLM about 10 minutes; planner and micro-experiments about 10 minutes; test
a few minutes. **About 20–30 minutes.**

Output: `runs/Set_3__self-learnt-retrieval/<timestamp>_<GPU>_rules10/` with `rulebooks/` (every version), `planner_round*.txt`, and
`results/summary.md`, `set10_rulebook.xlsx`.

**Runtime → Change runtime type → A100 GPU.**

In [1]:
# ==== settings ====
PROJECT_DIR = "/content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching"
EXPERIMENT  = "Set_3__self-learnt-retrieval"
ROUNDS = 4
NEW_RUN = False                       # False = continue the latest run of this notebook
DRY_RUN = False                       # True = notebook 27's dry run and a stand-in planner on CPU

EXPECTED_CODE = {
 "ssw.py": "c9fbdc7f6b91c58b607021b8be8c03ae2ff67dc1024fd2e24fffeb18e239ed98",
 "sweep.py": "9382292c6fe396cf79c73b212aaf6c2d677cae7f9311c3a5304f0578b52e6bb6",
 "routers.py": "8a293df1c8955716950f535cf9a3ced24439e00250173da1a3df110c0ed87304",
 "hidden.py": "574e6c1390e47c6de833a0c10b32c24eed0dee73a35eabc17f58842835e7f105",
 "final.py": "f18a6fba3d1bdd1f6700f6dc535f10ec0c41dbf5192b7b39ad097f8a1b88be53",
 "retriever_check.py": "fde08c549ecbd6bfc5527a42c731aa51dcff687a64818d68cf9cb30090e017e3",
 "ensemble.py": "5c5e87175d2484d806fda913a9555f71378a6031f050c37e89c20305942b76fc",
 "experts.py": "17ed39957e54853d178b47fa73ba328b50f0589b129ccb191a2efb0a2dea466b",
 "retft.py": "4a0d27d24459ba179c60dcce39a85285ae422dfa9e8d7cbc160e79a0a9932879",
 "casc8.py": "ba0d23a1aed58e9da5426406c0c953ae04b97c46cbaa820fa53328ea3e4c0bdd",
 "route9.py": "774600dad0bebf98f4957ab3131e23936973dc6cb117f0912cb4f01a5f96a83e",
 "rules10.py": "65d933f7fe05df7ae1858545444297df236efe6ea952dcf34d420d9ac1864d8c"
}

In [2]:
import subprocess, sys, os, json, shutil, time
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader 2>/dev/null",
                     shell=True, capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none")
if not DRY_RUN:
    assert gpu, "No GPU: Runtime -> Change runtime type -> A100 GPU (or set DRY_RUN = True)."
    if "T4" in gpu:
        raise SystemExit("This runtime has a T4, which cannot run this: the 7B model alone (15 GB) does not fit in its memory.\n"
                         "Runtime -> Change runtime type -> A100 GPU (or G4 / L4) -> Save, then run all cells again.")
    if "A100" not in gpu and "G4" not in gpu and "RTX PRO 6000" not in gpu:
        print("Note: planned for an A100 or G4; this GPU works but timings will differ.")
GPU_TAG = "cpu" if not gpu else gpu.split(",")[0].replace("NVIDIA", "").replace("Tesla", "").strip().split()[0].split("-")[0]
T_START = time.time()

GPU: Tesla T4, 15360 MiB, 7.5


SystemExit: This runtime has a T4, which cannot run this: the 7B model alone (15 GB) does not fit in its memory.
Runtime -> Change runtime type -> A100 GPU (or G4 / L4) -> Save, then run all cells again.

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### Check that Drive is serving the current code; find notebook 27's run

In [ ]:
import glob, hashlib
from datetime import datetime
from zoneinfo import ZoneInfo
def sha(p):
    return hashlib.sha256(open(p, "rb").read()).hexdigest()
CODE_DIR = "/content/set10code"
os.makedirs(CODE_DIR, exist_ok=True)
stale = []
for f, want in EXPECTED_CODE.items():
    shutil.copy(f"{PROJECT_DIR}/code/{f}", f"{CODE_DIR}/{f}")
    if sha(f"{CODE_DIR}/{f}") != want:
        stale.append(f)
assert not stale, (f"Drive is serving a different version of {stale} than this notebook was made for. Wait two minutes, then "
                   "Runtime -> Disconnect and delete runtime, and run again.")
routes = [d for d in sorted(glob.glob(f"{PROJECT_DIR}/runs/{EXPERIMENT}/*_route05" + ("_dryrun" if DRY_RUN else ""))) if os.path.exists(f"{d}/results/summary.md")]
assert routes, "Needs the finished run of notebook 27."
ROUTE_RUN = routes[-1]
CFG = json.load(open(f"{ROUTE_RUN}/config.json"))
SESSION = datetime.now(ZoneInfo("Asia/Singapore")).strftime("%Y%m%d-%H%M%S")
tag = "_rules10_dryrun" if DRY_RUN else "_rules10"
prev = sorted(glob.glob(f"{PROJECT_DIR}/runs/{EXPERIMENT}/*{tag}"))
RUN_DIR = prev[-1] if (prev and not NEW_RUN) else f"{PROJECT_DIR}/runs/{EXPERIMENT}/{SESSION}_{GPU_TAG}{tag}"
SESSION_DIR = f"{RUN_DIR}/sessions/{SESSION}"
for d in (f"{RUN_DIR}/code", f"{RUN_DIR}/logs", SESSION_DIR):
    os.makedirs(d, exist_ok=True)
for f in EXPECTED_CODE:
    shutil.copy(f"{CODE_DIR}/{f}", f"{RUN_DIR}/code/{f}")
TIMES = {}
print("Code files: current.")
print("Notebook 27 run:", ROUTE_RUN)
print("Run folder:     ", RUN_DIR)

In [ ]:
MODELS = {   # key: model, folder under models/, parameters (for compute), nominal size (for the AgentTTS budget formula)
    "q3":  {"id": "Qwen/Qwen2.5-3B-Instruct",        "dir": "Qwen2.5-3B-Instruct",        "params": 3.09e9,  "nominal": 3e9},
    "q7":  {"id": "Qwen/Qwen2.5-7B-Instruct",        "dir": "Qwen2.5-7B-Instruct",        "params": 7.62e9,  "nominal": 7e9},
    "q32": {"id": "Qwen/Qwen2.5-32B-Instruct-AWQ",   "dir": "Qwen2.5-32B-Instruct-AWQ",   "params": 32.76e9, "nominal": 32e9},
    "l3":  {"id": "meta-llama/Llama-3.2-3B-Instruct", "dir": "Llama-3.2-3B-Instruct",     "params": 3.21e9,  "nominal": 3e9},
    "l8":  {"id": "meta-llama/Llama-3.1-8B-Instruct", "dir": "Llama-3.1-8B-Instruct",     "params": 8.03e9,  "nominal": 8e9},
}
LOCAL = "/content/models"
for m in MODELS.values():
    m["path"] = f"{LOCAL}/{m['dir']}"

MODELS = {k: v for k, v in MODELS.items() if k in ("q32",)}
json.dump(MODELS, open(f"{RUN_DIR}/models.json", "w"), indent=1)
json.dump({"experiment": EXPERIMENT, "set": "10 self-learnt rulebook", "route_run": ROUTE_RUN, "rounds": ROUNDS, "gpu": gpu, "dry_run": DRY_RUN,
           "code": EXPECTED_CODE, "started": SESSION}, open(f"{RUN_DIR}/config.json", "w"), indent=1)

def minutes_left():
    return 999.0

def run_stage(args, log_name):
    cmd = [sys.executable, "-u", f"{CODE_DIR}/rules10.py"] + [str(a) for a in args]
    t0 = time.time()
    with open(f"{RUN_DIR}/logs/{log_name}", "a") as log, open(f"{SESSION_DIR}/{log_name}", "a") as slog:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); slog.write(line)
            if not any(k in line for k in ("it/s]", "s/it]", "Loading checkpoint", "INFO ", "(EngineCore", "WARNING ", "Warning", "warn", "Loading weights")):
                print(line, end="")
        p.wait()
    TIMES[log_name.replace(".log", "")] = TIMES.get(log_name.replace(".log", ""), 0) + round(time.time() - t0)
    json.dump(TIMES, open(f"{SESSION_DIR}/stage_times.json", "w"), indent=1)
    print(f"--- {log_name}: exit {p.returncode}, {time.time() - t0:.0f}s")
    assert p.returncode == 0, f"stage failed - see {RUN_DIR}/logs/{log_name}"

COMMON = ["--route_run", ROUTE_RUN, "--label_dir", CFG["label_dir"], "--old_data", CFG["old_data"], "--fresh_data", CFG["fresh_data"],
          "--old_sweep", CFG["old_sweep"], "--fresh_run", CFG["fresh_run"], "--out_dir", RUN_DIR, "--models_json", f"{RUN_DIR}/models.json"]
LEARNT = os.path.exists(f"{RUN_DIR}/rulebooks/rulebook_final.json")
print("rulebook already learnt:", LEARNT)

### 1. The planner model (Qwen 32B) and vLLM

In [ ]:
import importlib.util
if not LEARNT and not DRY_RUN:
    m = MODELS["q32"]
    if not os.path.exists(f"{m['path']}/config.json"):
        t = time.time()
        print("copying Qwen 32B to local disk ...")
        shutil.copytree(f"{PROJECT_DIR}/models/{m['dir']}", m["path"] + ".part", dirs_exist_ok=True)
        os.rename(m["path"] + ".part", m["path"])
        TIMES["copy_q32"] = round(time.time() - t)
    if importlib.util.find_spec("vllm") is None:
        t = time.time()
        !pip install -q vllm
        !pip uninstall -y -q torchaudio   # Colab's torchaudio is built for another CUDA version and breaks imports
        chk = subprocess.run([sys.executable, "-c", "from vllm import LLM; import vllm, torch; print('vLLM', vllm.__version__, '| torch', torch.__version__)"],
                             capture_output=True, text=True)
        print(chk.stdout or chk.stderr[-2000:])
        assert chk.returncode == 0, "vLLM does not import - Runtime -> Disconnect and delete runtime, then run again."
        TIMES["install_vllm"] = round(time.time() - t)

### 2. Micro-experiments on the dev questions: learn the rulebook

In [ ]:
if not LEARNT:
    run_stage(["learn", "--rounds", ROUNDS, "--planner", "mock" if DRY_RUN else "vllm"] + COMMON, "learn.log")
print(open(f"{RUN_DIR}/rulebooks/rulebook_final.json").read())

### 3. Test: the frozen rulebook on the 400 test questions (CPU)

In [ ]:
from IPython.display import Markdown, display
run_stage(["test"] + COMMON, "test.log")
display(Markdown(open(f"{RUN_DIR}/results/summary.md").read()))
print("Excel workbook:", f"{RUN_DIR}/results/set10_rulebook.xlsx")
TIMES["total_session"] = round(time.time() - T_START)
json.dump({"session": SESSION, "run_dir": RUN_DIR, "gpu": gpu, "seconds": TIMES, "code": EXPECTED_CODE, "complete": True},
          open(f"{SESSION_DIR}/session.json", "w"), indent=1)
print("Wall-clock this session (s):", TIMES)
print("\nALL DONE.")
try:
    drive.flush_and_unmount()
    print("Saved to Drive.")
except Exception as e:
    print("Drive flush skipped:", e)